In [ ]:
!apt-get install -y tesseract-ocr
!pip install -q gradio opencv-python pytesseract PyMuPDF nltk

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (5.3.4-1build5).
0 upgraded, 0 newly installed, 0 to remove and 0 not upgraded.


In [5]:
import gradio as gr
import cv2
import pytesseract
import fitz
import re
import nltk
import numpy as np
import tempfile
import os
from PIL import Image
from nltk.tokenize import wordpunct_tokenize

document_rules = {
    "Student Registration Form": {
        "Student Name": r"\bname\b",
        "Date of Birth": r"date of birth|\bdob\b",
        "Student ID": r"student id|roll no",
        "Email": r"\bemail\b",
        "Phone Number": r"phone|mobile",
        "Address": r"address",
        "Signature": r"signature"
    },
    "Job Application Form": {
        "Applicant Name": r"applicant name|full name",
        "Email": r"\bemail\b",
        "Phone Number": r"phone|mobile",
        "Education": r"education|qualification",
        "Work Experience": r"experience",
        "Skills": r"\bskills\b",
        "Address": r"address",
        "Signature": r"signature"
    },
    "Leave Application": {
        "Student/Employee Name": r"name",
        "Date": r"\bdate\b",
        "Reason": r"reason",
        "From Date": r"from date|start date",
        "To Date": r"to date|end date",
        "Contact Number": r"phone|mobile|contact",
        "Signature": r"signature"
    }
}

In [9]:
def extract_text(path):
    text = ""
    if path.lower().endswith(".pdf"):
        pdf = fitz.open(path)
        for page in pdf:
            page_text = page.get_text()
            if page_text.strip():
                text += page_text + "\n"
            else:
                pix = page.get_pixmap(matrix=fitz.Matrix(2, 2))
                img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
                img = cv2.cvtColor(np.array(img), cv2.COLOR_RGB2BGR)
                gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
                text += pytesseract.image_to_string(gray) + "\n"
    else:
        img = cv2.imread(path)
        if img is None:
            raise ValueError("Unable to read this image.")
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        text = pytesseract.image_to_string(gray)
    return text
def verify_document(file, doc_type):
    if file is None:
        return "Please upload a document."
    try:
        text = extract_text(file)
        tokens = wordpunct_tokenize(text.lower())
        clean_text = " ".join(tokens)
        rules = document_rules[doc_type]
        results = []
        found = 0
        for field, pattern in rules.items():
            status = "Found" if re.search(pattern, clean_text) else "Missing"
            results.append((field, status))
            if status == "Found":
                found += 1
        score = int(found / len(rules) * 100)
        report = "SMART DOCUMENT VERIFICATION REPORT\n"
        report += "=" * 35 + "\n"
        report += "Document Type: " + doc_type + "\n\n"
        report += "FIELD VERIFICATION:\n"
        for field, status in results:
            report += field + " : " + status + "\n"
        report += "\nCOMPLETENESS SCORE: " + str(score) + "%\n"
        report += "STATUS: " + ("Complete" if found == len(rules) else "Needs Correction") + "\n"
        report += "\nFORMAT VALIDATION:\n"
        email = re.findall(r"[\w.+-]+@[\w-]+\.[\w.-]+", text)
        phone = re.findall(r"\b[6-9]\d{9}\b", text)
        dates = re.findall(r"\b\d{1,2}[-/]\d{1,2}[-/]\d{2,4}\b", text)
        report += "Email Format: " + ("Valid" if email else "Not Found or Invalid") + "\n"
        report += "Phone Format: " + ("Valid" if phone else "Not Found or Invalid") + "\n"
        report += "Date Format: " + ("Found" if dates else "Not Found") + "\n"
        report += "\nSUGGESTIONS:\n"
        missing = [field for field, status in results if status == "Missing"]
        if missing:
            for field in missing:
                report += "Please provide " + field + ".\n"
        else:
            report += "All required fields were found.\n"
        if not email:
            report += "Please check the email address format.\n"
        if not phone:
            report += "Please check the phone number format.\n"
        report += "\nNOTE: This tool checks document completeness, not document authenticity."
        return report
    except Exception as e:
        return "Error: " + str(e)

In [10]:
with gr.Blocks(theme=gr.themes.Soft()) as app:
    gr.Markdown("# Smart Document Verification Assistant")
    gr.Markdown("Upload a document to check missing and incomplete information.")
    doc_type = gr.Dropdown(
        choices=list(document_rules.keys()),
        value="Student Registration Form",
        label="Select Document Type"
    )
    file = gr.File(
        label="Upload PDF or Image",
        type="filepath",
        file_types=[".pdf", ".png", ".jpg", ".jpeg"]
    )
    button = gr.Button("Verify Document")
    output = gr.Textbox(label="Verification Report", lines=18)
    button.click(
        verify_document,
        inputs=[file, doc_type],
        outputs=output
    )
app.launch(share=True)

/tmp/ipykernel_596/3255627654.py:1: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as app:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://c7546e44fe4459c102.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
